# Non-Linear Variable Relationships in Census Data

This notebook investigates pairwise relationships between 408 census variables using:
- **Spearman correlation** (rank-based, captures monotonic non-linear relationships)
- **Pearson correlation** as linear baseline
- Focus on "purely non-linear" relationships (high |Spearman|, low |Pearson|)

**Why Spearman vs Pearson?**
- When |Spearman| >> |Pearson|, indicates a **monotonic non-linear relationship** (e.g., exponential, logarithmic)
- Both correlations are computed via fast vectorized operations (~30 seconds total)
- Alternative to Mutual Information which takes 2-4 hours

**Analysis Pipeline:**
1. Compute pairwise Spearman and Pearson correlations for all variable pairs
2. Identify relationships with high |Spearman| but low |Pearson|
3. Visualize top-k most non-linear relationships with scatter plots and fitted curves

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import mutual_info_regression
from scipy.stats import pearsonr
from scipy.optimize import curve_fit
import pickle

# Set plotting style
plt.style.use('default')
sns.set_palette('husl')

## 2. Configuration

In [ ]:
# Random seed for reproducibility
random_seed = 20210321
np.random.seed(random_seed)

# Paths
cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
category_path = "../data/census_data/Category.csv"

# Output directories
output_dir = "./plots/nonlinear_relationships/"
cache_dir = "../data/cache/"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/data/", exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

# Analysis parameters
use_cache = True  # Set to False to force recomputation
n_top_relationships = 50  # Top-k relationships to visualize
subsample_for_plots = 10000  # Subsample OAs for scatter plots to reduce clutter
pearson_threshold = 0.3  # Low linear correlation threshold

print(f"Configuration:")
print(f"  Random seed: {random_seed}")
print(f"  Use cache: {use_cache}")
print(f"  Top relationships to visualize: {n_top_relationships}")
print(f"  Output directory: {output_dir}")
print(f"  Pearson threshold: {pearson_threshold}")

## 3. Load and Prepare Data

In [ ]:
# Load census data
df = pd.read_parquet(cleaned_data_path)
df = df.set_index('OA')
print(f"Census data: {df.shape}")
print(f"Variables: {len(df.columns)}")
print(f"Output Areas: {len(df)}")

# Load category mapping
categories = pd.read_csv(category_path)
print(f"\nCategory mapping loaded: {len(categories)} tables")

# Create variable-to-category mapping
var_to_category = {}
for var in df.columns:
    prefix = var[:5]  # e.g., 'ts001', 'ts002'
    matching_cat = categories[categories['Table'] == prefix]
    if len(matching_cat) > 0:
        var_to_category[var] = matching_cat['Category'].values[0]
    else:
        var_to_category[var] = 'Unknown'

# Print category distribution
cat_counts = pd.Series(var_to_category).value_counts()
print(f"\nVariables by category:")
print(cat_counts)

## 4. Compute Pairwise Metrics (Fast Method)

This section computes:
1. **Pearson correlation** (linear relationship) - vectorized, fast
2. **Spearman correlation** (rank-based, captures monotonic non-linear) - vectorized, fast
3. **Non-linearity score** = |Spearman| - |Pearson|

**Why this is fast**: Both Pearson and Spearman can be computed in a single vectorized operation (~30 seconds total).

Results are cached to avoid recomputation.

In [ ]:
cache_file = f"{cache_dir}/pairwise_metrics_spearman.parquet"

if use_cache and os.path.exists(cache_file):
    print(f"Loading cached pairwise metrics from {cache_file}")
    pairwise_df = pd.read_parquet(cache_file)
    print(f"Loaded {len(pairwise_df)} pairwise relationships")
else:
    print("Computing pairwise metrics (Spearman vs Pearson)...")
    
    # STEP 1: Filter out constant/near-constant variables
    var_variance = df.var()
    valid_vars = var_variance[var_variance > 1e-6].index.tolist()
    df_filtered = df[valid_vars]
    print(f"Filtered to {len(valid_vars)} variables (removed {len(df.columns) - len(valid_vars)} constant variables)")
    
    # STEP 2: Compute Pearson correlation matrix (vectorized - FAST)
    print("\nComputing Pearson correlation matrix...")
    pearson_matrix = df_filtered.corr(method='pearson')
    print(f"Pearson correlation matrix: {pearson_matrix.shape}")
    
    # STEP 3: Compute Spearman correlation matrix (vectorized - FAST)
    print("Computing Spearman correlation matrix...")
    spearman_matrix = df_filtered.corr(method='spearman')
    print(f"Spearman correlation matrix: {spearman_matrix.shape}")
    
    # STEP 4: Extract upper triangle and create pairwise DataFrame
    print("\nCreating pairwise relationship DataFrame...")
    relationships = []
    
    for i in range(len(valid_vars)):
        for j in range(i + 1, len(valid_vars)):
            var1 = valid_vars[i]
            var2 = valid_vars[j]
            
            pearson_r = pearson_matrix.iloc[i, j]
            spearman_r = spearman_matrix.iloc[i, j]
            
            relationships.append({
                'var1': var1,
                'var2': var2,
                'var1_category': var_to_category.get(var1, 'Unknown'),
                'var2_category': var_to_category.get(var2, 'Unknown'),
                'pearson_r': pearson_r,
                'spearman_r': spearman_r,
                'pearson_r_abs': abs(pearson_r),
                'spearman_r_abs': abs(spearman_r),
            })
    
    pairwise_df = pd.DataFrame(relationships)
    
    # STEP 5: Compute non-linearity score
    # |Spearman| > |Pearson| indicates monotonic non-linear relationship
    pairwise_df['nonlinearity_score'] = pairwise_df['spearman_r_abs'] - pairwise_df['pearson_r_abs']
    
    # Save to cache
    pairwise_df.to_parquet(cache_file)
    print(f"\nSaved {len(pairwise_df)} pairwise relationships to {cache_file}")

print(f"\nPairwise metrics computed:")
print(f"  Total pairs: {len(pairwise_df)}")
print(f"  Pearson |r| range: [{pairwise_df['pearson_r_abs'].min():.4f}, {pairwise_df['pearson_r_abs'].max():.4f}]")
print(f"  Spearman |r| range: [{pairwise_df['spearman_r_abs'].min():.4f}, {pairwise_df['spearman_r_abs'].max():.4f}]")
print(f"  Non-linearity score range: [{pairwise_df['nonlinearity_score'].min():.4f}, {pairwise_df['nonlinearity_score'].max():.4f}]")

## 5. Filter and Rank Non-Linear Relationships

Identify "purely non-linear" relationships:
- High |Spearman| (strong monotonic relationship)
- Low |Pearson| (weak linear relationship)
- Positive non-linearity score (|Spearman| > |Pearson|)

In [ ]:
# Define thresholds for "purely non-linear" relationships
spearman_threshold_percentile = 75  # Top 25% Spearman correlations
spearman_threshold = pairwise_df['spearman_r_abs'].quantile(spearman_threshold_percentile / 100)

# Filter criteria:
# 1. High |Spearman| (top quartile) - strong monotonic relationship
# 2. Low |Pearson| (< 0.3) - weak linear relationship
# 3. Positive non-linearity score

nonlinear_candidates = pairwise_df[
    (pairwise_df['spearman_r_abs'] > spearman_threshold) &
    (pairwise_df['pearson_r_abs'] < pearson_threshold) &
    (pairwise_df['nonlinearity_score'] > 0)
].copy()

print(f"Filtering for 'purely non-linear' relationships:")
print(f"  |Spearman| threshold (75th percentile): {spearman_threshold:.4f}")
print(f"  |Pearson| threshold: {pearson_threshold}")
print(f"  Found: {len(nonlinear_candidates)} candidate pairs")

# Sort by non-linearity score
nonlinear_candidates = nonlinear_candidates.sort_values('nonlinearity_score', ascending=False)

# Select top-k for visualization
top_nonlinear = nonlinear_candidates.head(n_top_relationships)

print(f"\nTop {n_top_relationships} most non-linear relationships:")
print(top_nonlinear[['var1', 'var2', 'spearman_r', 'pearson_r', 'nonlinearity_score']].head(20))

# Save results
top_nonlinear.to_csv(f"{output_dir}/data/top_nonlinear_relationships.csv", index=False)
nonlinear_candidates.to_csv(f"{output_dir}/data/all_nonlinear_candidates.csv", index=False)
pairwise_df.to_csv(f"{output_dir}/data/all_pairwise_metrics.csv", index=False)
print(f"\nResults saved to {output_dir}/data/")

## 6. Overview Visualizations

### 6.1 Spearman vs Pearson Scatter Plot

In [ ]:
# Scatter plot: |Spearman| vs |Pearson| for all pairs
fig, ax = plt.subplots(figsize=(10, 8))

# Sample for visualization if too many points
if len(pairwise_df) > 50000:
    plot_df = pairwise_df.sample(n=50000, random_state=random_seed)
else:
    plot_df = pairwise_df

scatter = ax.scatter(
    plot_df['pearson_r_abs'], 
    plot_df['spearman_r_abs'],
    c=plot_df['nonlinearity_score'],
    cmap='RdYlGn',
    alpha=0.3,
    s=10
)

# Highlight top non-linear relationships
ax.scatter(
    top_nonlinear['pearson_r_abs'],
    top_nonlinear['spearman_r_abs'],
    color='red',
    s=30,
    alpha=0.7,
    label=f'Top {n_top_relationships} non-linear',
    edgecolors='black',
    linewidths=0.5
)

# Add diagonal line (Spearman = Pearson means linear)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=2, label='Linear (Spearman=Pearson)')

ax.set_xlabel('|Pearson Correlation|', fontsize=14)
ax.set_ylabel('|Spearman Correlation|', fontsize=14)
ax.set_title('Spearman vs Pearson Correlation\\nfor Census Variable Pairs', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add reference lines
ax.axvline(x=pearson_threshold, color='gray', linestyle='--', alpha=0.5, linewidth=2)
ax.axhline(y=spearman_threshold, color='gray', linestyle='--', alpha=0.5, linewidth=2)
ax.legend(fontsize=12)

plt.colorbar(scatter, ax=ax, label='Non-linearity Score')
plt.tight_layout()
plt.savefig(f"{output_dir}/spearman_vs_pearson_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

### 6.2 Category-Based Heatmap

In [ ]:
# Heatmap: Average |Spearman| by variable category pairs
category_spearman = pairwise_df.groupby(['var1_category', 'var2_category'])['spearman_r_abs'].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    category_spearman, 
    annot=True, 
    fmt='.3f', 
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Average |Spearman Correlation|'}
)
ax.set_title('Average |Spearman Correlation| Between Census Categories', fontsize=16, fontweight='bold')
ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{output_dir}/category_spearman_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

### 6.3 Distribution Plots

In [ ]:
# Distribution of Pearson, Spearman, and Non-linearity scores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(pairwise_df['pearson_r_abs'], bins=100, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0].axvline(pearson_threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {pearson_threshold}')
axes[0].set_xlabel('|Pearson Correlation|', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of |Pearson r|', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(pairwise_df['spearman_r_abs'], bins=100, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].axvline(spearman_threshold, color='red', linestyle='--', linewidth=2, label=f'75th percentile: {spearman_threshold:.3f}')
axes[1].set_xlabel('|Spearman Correlation|', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of |Spearman r|', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].hist(pairwise_df['nonlinearity_score'], bins=100, color='lightgreen', edgecolor='black', alpha=0.7)
axes[2].axvline(0, color='gray', linestyle='--', linewidth=2, alpha=0.5)
axes[2].set_xlabel('Non-linearity Score (|Spearman| - |Pearson|)', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_title('Distribution of Non-linearity Scores', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{output_dir}/distribution_plots.png", dpi=300, bbox_inches='tight')
plt.show()

## 7. Top-K Relationship Scatter Plots

Visualize the top non-linear relationships with scatter plots and polynomial curve fits.

In [ ]:
# Function to fit curves and visualize non-linear relationships
def fit_and_plot_nonlinear(var1, var2, data, ax, rank):
    """
    Fit polynomial and plot scatter with curve overlay
    """
    # Subsample for visualization
    if len(data) > subsample_for_plots:
        plot_data = data.sample(n=subsample_for_plots, random_state=random_seed)
    else:
        plot_data = data
    
    x = plot_data[var1].values
    y = plot_data[var2].values
    
    # Remove any NaN/inf values
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.3, s=5, color='steelblue')
    
    # Fit polynomial (degree 3)
    try:
        # Sort for smooth curve plotting
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y[sort_idx]
        
        # Fit polynomial
        coeffs = np.polyfit(x_sorted, y_sorted, deg=3)
        poly_fn = np.poly1d(coeffs)
        
        # Generate smooth curve
        x_smooth = np.linspace(x.min(), x.max(), 200)
        y_smooth = poly_fn(x_smooth)
        
        ax.plot(x_smooth, y_smooth, color='red', linewidth=2, label='Polynomial fit (deg=3)')
    except:
        pass  # Skip fitting if fails
    
    # Get metrics
    row = top_nonlinear[(top_nonlinear['var1'] == var1) & (top_nonlinear['var2'] == var2)].iloc[0]
    spearman = row['spearman_r']
    pearson = row['pearson_r']
    nl_score = row['nonlinearity_score']
    
    ax.set_xlabel(f'{var1}\\n({var_to_category.get(var1, "Unknown")})', fontsize=9)
    ax.set_ylabel(f'{var2}\\n({var_to_category.get(var2, "Unknown")})', fontsize=9)
    ax.set_title(f'Rank {rank}: ρ={spearman:.3f}, r={pearson:.3f}, NL={nl_score:.3f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

# Create grid of scatter plots for top relationships
n_cols = 5
n_rows = (n_top_relationships + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for rank, (idx, row) in enumerate(top_nonlinear.iterrows(), 1):
    if rank > n_top_relationships:
        break
    
    ax = axes[rank - 1]
    fit_and_plot_nonlinear(row['var1'], row['var2'], df, ax, rank)

# Hide unused subplots
for i in range(n_top_relationships, len(axes)):
    axes[i].axis('off')

plt.suptitle(f'Top {n_top_relationships} Non-Linear Relationships in Census Variables', 
             fontsize=18, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(f"{output_dir}/top_nonlinear_scatterplots.png", dpi=300, bbox_inches='tight')
plt.show()

## 8. Summary Statistics and Report

In [ ]:
print("="*80)
print("NON-LINEAR RELATIONSHIP ANALYSIS SUMMARY")
print("="*80)

print(f"\nDataset:")
print(f"  Variables analyzed: {len(df.columns)}")
print(f"  Output Areas: {len(df)}")
print(f"  Total pairwise relationships: {len(pairwise_df)}")

print(f"\nSpearman Correlation Statistics:")
print(f"  Mean |Spearman|: {pairwise_df['spearman_r_abs'].mean():.4f}")
print(f"  Std |Spearman|: {pairwise_df['spearman_r_abs'].std():.4f}")
print(f"  75th percentile: {spearman_threshold:.4f}")

print(f"\nPearson Correlation Statistics:")
print(f"  Mean |r|: {pairwise_df['pearson_r_abs'].mean():.4f}")
print(f"  Std |r|: {pairwise_df['pearson_r_abs'].std():.4f}")

print(f"\nNon-Linear Relationships:")
print(f"  Candidates (|Spearman| > p75, |r| < {pearson_threshold}): {len(nonlinear_candidates)}")
print(f"  Percentage of all pairs: {len(nonlinear_candidates) / len(pairwise_df) * 100:.2f}%")

print(f"\nTop 10 Most Non-Linear Pairs:")
for idx, row in enumerate(top_nonlinear.head(10).itertuples(), 1):
    print(f"  {idx}. {row.var1} ↔ {row.var2}")
    print(f"      ρ={row.spearman_r:.3f}, r={row.pearson_r:.3f}, NL_score={row.nonlinearity_score:.3f}")

print(f"\nCategory Pairs with Highest Average |Spearman|:")
top_cat_pairs = pairwise_df.groupby(['var1_category', 'var2_category'])['spearman_r_abs'].mean().nlargest(5)
for (cat1, cat2), spearman in top_cat_pairs.items():
    print(f"  {cat1} ↔ {cat2}: {spearman:.3f}")

print("\n" + "="*80)
print(f"Analysis complete! Outputs saved to: {output_dir}")
print("="*80)